In [2]:
from IPython.display import display, HTML
display(HTML ("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.text_cell_render.rendered_html{font-size:12pt;}""
div.output {font-size:12pt; font-weight:bold;}
div.input{font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# 벡터DB : Chroma vs. Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB
- Pinecone : 클라우드 vector DB
    (https://www.pinecone.io에서 api key 생성 -> .env에 추가(PINECONE_API_KEY등록)

# 0. 패키지 설치

In [3]:
%pip install -q pinecone langchain-pinecone --no-warn-script-location

Note: you may need to restart the kernel to use updated packages.


# 1. knowledge Base 구성을 위한 데이터 생성

In [4]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('data/소득세법(법률)(제21065호)(20260102).docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, 
    chunk_overlap=200,
    # separators=["\n\n", "\n", " ", ""]
)
document_list = loader.load_and_split(text_splitter=text_splitter)
len(document_list)

193

In [5]:
# embedding : OpenAI API text-embedding-3-Large
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding = OpenAIEmbeddings(model='text-embedding-3-large')

In [7]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os
pc = Pinecone(
    api_key=os.getenv("PINECONE_API_KEY")
)
# 데이터를 처음 업로드할 때
# index_name = "tax-index"
# database = PineconeVectorStore.from_documents(
#     documents=document_list,
#     embedding=embedding,
#     index_name=index_name
# )
# 업로드한 벡터db를 가져올 때
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name
)

CPU times: total: 0 ns
Wall time: 996 μs


# 2. 답변 생성을 위한 Retrieval

In [8]:
query = "연봉이 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query, k=3)

In [9]:
# retrieved_docs[2].page_content
retrieved_doc = "\n\n---\n\n".join([doc.page_content for doc in retrieved_docs])

In [11]:
retriever = database.as_retriever(
    search_kwargs={"k":3}
)
retrieved_docs = retriever.invoke(query)

# 3. 답변 생성

In [12]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model = "gpt-4.1-nano")

In [14]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(
    model = "solar-pro2",
    reasoning_effort="high" #느리지만 더 깊게 추론함(low, medium)
)

In [17]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다.
- [context]를 참고해서 사용자의 질문에 답변해 주세요.
- [context]는 다음과 같아요
{retrieved_doc}
- 질문 : {query}"""

In [18]:
ai_message = llm.invoke(prompt)

In [19]:
print(ai_message.content)

소득세법 조항을 기반으로 연봉 5천만원인 직장인의 소득세를 계산하는 과정은 다음과 같습니다. 다만, 제공된 [context]에는 **세율표**나 **근로소득공제율** 등 구체적인 수치가 명시되어 있지 않아, 일반적인 국세청 기준을 참고하여 설명합니다. 실제 계산 시에는 최신 세법과 시행령을 반드시 확인해야 합니다.

---

### **1. 근로소득 금액 산정**
- **총급여액**: 5,000만원 (비과세소득 제외).  
- **근로소득공제 적용**:  
  총급여액에 따라 공제율이 차등 적용됩니다(2023년 기준).  
  - 500만원 이하: 70% 공제  
  - 500만원 초과 ~ 1,500만원: 40% 공제  
  - 1,500만원 초과 ~ 4,500만원: 15% 공제  
  - 4,500만원 초과 ~ 1억원: 5% 공제  
  - 1억원 초과: 2% 공제  

  **계산 예시**:  
  ```
  (500만원 × 70%) + (1,000만원 × 40%) + (3,000만원 × 15%) + (500만원 × 5%)  
  = 350만원 + 400만원 + 450만원 + 25만원 = **1,225만원 공제**  
  ```
  - **과세표준**: 5,000만원 - 1,225만원 = **3,775만원**

---

### **2. 산출세액 계산 (누진세율 적용)**
과세표준에 따라 다음과 같은 세율이 적용됩니다(2023년 기준):  
| 과세표준 (만원) | 세율 | 누진공제 (만원) |
|----------------|------|------------------|
| ~ 1,200        | 6%   | -                |
| 1,200 ~ 4,600  | 15%  | 108              |
| 4,600 ~ 8,800  | 24%  | 522              |
| 8,800 ~ 1.5억  | 35%  | 1,490            |
| 1.5억 ~ 3억    | 38%  | 1,940          